# Part 2: held-out comparison at strength 1.0, 2.0 and 3.0

Five models on two planes at three separability settings, scored on seeds 5-44. Hyperparameters
come from the selection notebook, which selects on seeds 0-4 independently at each strength, so
nothing reported here benefited from the data used to tune it.

All three LR forms are tuned within their own form, so the complexity ladder
(`lr_interaction - lr_linear`, then `lr_quadratic - lr_interaction`) compares tuned against
tuned. `lr_linear` remains the paired baseline.

In [ ]:
import os, sys, json, time, copy, itertools, subprocess, pathlib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

REPO_URL = "https://github.com/spragunr/maser-research"
REPO_DIR = "maser-research"
PIN      = None          # set to a commit SHA to freeze the generator

try:
    _ip = str(get_ipython())
except NameError:
    _ip = ""

if "google.colab" in _ip and os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    # fetch+reset, not a bare existence check: a stale clone must not survive
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--quiet"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    PIN or "origin/main", "--quiet"], check=True)
    os.chdir(REPO_DIR)

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
for _p in ("src", "../src"):
    _ap = os.path.abspath(_p)
    if os.path.isdir(_ap) and _ap not in sys.path:
        sys.path.insert(0, _ap)

# ONE import path for the generator. Do not also import src.synth_data:
# that creates a second module object for the same file.
import synth_data as sd
from maser_data import FEATURES, TARGET

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import warnings
warnings.filterwarnings("ignore")

# ---- global config: identical for every model, plane and strength ----
SCENARIOS  = ["linear", "wedge", "box", "interaction", "blob"]
STRENGTHS  = [1.0, 2.0, 3.0]
PLANES     = ["xray", "wise"]
N_AT       = 50

N_SPLITS     = 5
N_REPEATS    = 1
SELECT_SEEDS = range(0, 5)    # selection notebook only
REPORT_SEEDS = range(5, 45)   # everything here (40 seeds)

# Smoke-test knob: set to e.g. 5 to rehearse the whole notebook in a few minutes.
SEED_LIMIT = None

def report_seeds():
    s = list(REPORT_SEEDS)
    return s[:SEED_LIMIT] if SEED_LIMIT else s

N_JOBS  = -1                  # outer-loop parallelism; models stay single-threaded
WISE_N  = None                # full WISE catalog

PLANE_COLS = {p: (FEATURES[p], TARGET[p]) for p in PLANES}

PRIMARY_METRICS    = ["prec_at_n", "mse_vs_truth"]
DIAGNOSTIC_METRICS = ["brier", "pr_auc", "auc"]
METRICS            = PRIMARY_METRICS + DIAGNOSTIC_METRICS
HIGHER_BETTER      = {"auc": True, "pr_auc": True, "prec_at_n": True,
                      "brier": False, "mse_vs_truth": False}
BASELINE_MODEL     = "lr_linear"
MODEL_ORDER        = ["lr_linear", "lr_interaction", "lr_quadratic", "rf", "hgb"]

# the LR complexity ladder: each step must beat the step below it, not just the baseline
LADDER = [("lr_interaction", "lr_linear"),
          ("lr_quadratic",   "lr_interaction"),
          ("lr_quadratic",   "lr_linear")]

def make_catalog(plane, scenario, strength, seed):
    """One catalog. WISE takes an explicit n; X-ray uses the repo default."""
    if plane == "wise":
        return sd.make_dataset("wise", scenario=scenario, strength=strength,
                               n=WISE_N, seed=seed)
    return sd.make_dataset(plane, scenario=scenario, strength=strength, seed=seed)

def dataset_fn(plane, scenario, strength):
    return lambda seed: make_catalog(plane, scenario, strength, seed)

print(f"report seeds: {report_seeds()[0]}-{report_seeds()[-1]} ({len(report_seeds())} seeds)"
      f" x {N_REPEATS} repeats x {N_SPLITS} folds")
print(f"strengths   : {STRENGTHS}")
print(f"planes      : {PLANES}")
for p in PLANES:
    f, t = PLANE_COLS[p]
    print(f"{p:5s} features: {f} | target: {t}")

# ---- provenance + guards: run first, paste the output when comparing runs ----
try:
    _sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                   text=True).strip()
except Exception:
    _sha = "unknown"

_chk = make_catalog("xray", "blob", 3.0, 0)
print("synth_data :", sd.__file__)
print("repo sha   :", _sha)
print("ceiling    :", _chk.attrs.get("ceiling"))
assert _chk.attrs.get("ceiling") == 0.4, \
    "generator is not at ceiling=0.4 - you are on a stale copy of synth_data.py"

# ceiling and the calibration reference must hold at EVERY strength, not just 3.0.
# mse_vs_truth is measured against p_true, which is only a valid calibration
# reference while p_obs == p_true (no distance noise). Fail loudly if that changes.
for plane, st, sc in itertools.product(PLANES, STRENGTHS, SCENARIOS):
    d = make_catalog(plane, sc, st, 0)
    assert d["p_true"].max() <= 0.4, f"{plane}/{sc}/s={st}: p_true exceeds the ceiling"
    assert np.allclose(d["p_true"], d["p_obs"]), (
        f"{plane}/{sc}/s={st}: p_obs != p_true - point mse_vs_truth at p_obs "
        "before trusting calibration numbers")
print("\nprovenance OK; ceiling and calibration reference hold at all three strengths")

## 2. Model builders

Byte-identical to the selection notebook, so a config selected there is the estimator scored here.

`features="interaction"` is `PolynomialFeatures(2, interaction_only=True)`: on a two-column plane
that is `[x1, x2, x1*x2]`, three terms. `"quadratic"` adds the two squares, five terms. The gap
between them is the only evidence that the squared terms are doing work.

In [ ]:
def make_lr(features="plain", C=1.0):
    """features: plain | interaction | quadratic. Solver is lbfgs, which leaves
    the intercept unpenalized - liblinear shrinks it and inflates the promised
    totals at low prevalence."""
    def build():
        steps = [StandardScaler()]
        if features == "interaction":
            steps.append(PolynomialFeatures(2, interaction_only=True, include_bias=False))
        elif features == "quadratic":
            steps.append(PolynomialFeatures(2, interaction_only=False, include_bias=False))
        steps.append(LogisticRegression(C=C, solver="lbfgs", max_iter=5000))
        return make_pipeline(*steps)
    return build

def make_rf(n_estimators=150, max_depth=4, min_samples_leaf=5, max_features="sqrt"):
    """max_features was swept during selection: on a two-column plane 'sqrt' means one candidate
    feature per split, and None gives bagged trees."""
    return lambda: RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_leaf=min_samples_leaf, max_features=max_features,
        random_state=0, n_jobs=1)

def make_hgb(max_depth=2, learning_rate=0.05, min_samples_leaf=20, max_iter=200):
    """sklearn's histogram GBT, so the notebook has no dependency that can differ between
    machines. early_stopping is off, so max_iter is used in full and the fit is deterministic."""
    return lambda: HistGradientBoostingClassifier(
        max_iter=max_iter, max_depth=max_depth, learning_rate=learning_rate,
        min_samples_leaf=min_samples_leaf, early_stopping=False, random_state=0)

def build_from_spec(spec):
    s = dict(spec)
    return {"lr": make_lr, "rf": make_rf, "hgb": make_hgb}[s.pop("kind")](**s)

## 3. Per-strength lineups

Paste the block printed by the selection notebook's last cell into the cell below, replacing the
two `None` lines. If `selection_results.json` is sitting in the working directory, leaving them
as `None` loads it instead. Either way the loader refuses to run on a partial lineup, so nothing
downstream can quietly fall back to another strength's hyperparameters.

In [ ]:
# ==================== PASTE THE SELECTION BLOCK BELOW ====================
LINEUPS       = None
LINEUP_SOURCE = None
# ==================== END PASTE ====================

SELECTION_JSON = "selection_results.json"

In [ ]:
def _load_lineups():
    if LINEUPS is not None:
        src = LINEUP_SOURCE or {st: "pasted, source not recorded" for st in STRENGTHS}
        return copy.deepcopy(LINEUPS), dict(src)
    p = pathlib.Path(SELECTION_JSON)
    if not p.exists():
        raise RuntimeError(
            f"no lineups: paste the selection block above, or put {SELECTION_JSON} in {os.getcwd()}")
    art = json.loads(p.read_text())
    return ({float(k): v for k, v in art["lineups"].items()},
            {float(k): v for k, v in art["source"].items()})

SPECS, SPEC_SOURCE = _load_lineups()

assert set(SPECS) == set(STRENGTHS), f"lineups cover {sorted(SPECS)}, need {STRENGTHS}"
for st in STRENGTHS:
    assert set(SPECS[st]) == set(PLANES), f"strength {st}: planes {sorted(SPECS[st])}"
    for p in PLANES:
        assert list(SPECS[st][p]) == MODEL_ORDER, \
            f"lineup {p}/s={st} does not match MODEL_ORDER"
        for m, spec in SPECS[st][p].items():
            assert spec.get("kind") in ("lr", "rf", "hgb"), f"{p}/s={st}/{m}: bad spec {spec}"
# a lineup reused across strengths answers a weaker question than the one asked; say so loudly
_dupes = [(a, b) for a, b in itertools.combinations(STRENGTHS, 2) if SPECS[a] == SPECS[b]]
if _dupes:
    print("!" * 78)
    print(f"WARNING: identical lineups at strengths {_dupes}. If that is a real selection "
          "result it is fine; if it is a copy, the cross-strength claim is not supported.")
    print("!" * 78, "\n")

TUNED = {st: {p: {m: build_from_spec(sp) for m, sp in d.items()}
              for p, d in pl.items()}
         for st, pl in SPECS.items()}

def lineup_table():
    rows = []
    for st in STRENGTHS:
        for p in PLANES:
            for m in MODEL_ORDER:
                s = dict(SPECS[st][p][m])
                rows.append({"strength": st, "plane": p, "model": m,
                             "kind": s.pop("kind"),
                             "params": ", ".join(f"{k}={v}" for k, v in s.items()),
                             "source": SPEC_SOURCE[st]})
    return pd.DataFrame(rows).set_index(["strength", "plane", "model"])

display(lineup_table())

## 4. Evaluation harness

One definition of every metric, used everywhere, and the same one the selection notebook used.

Per (plane, strength, scenario, seed): the catalog is generated **once** and shared by every
model, and each model sees the identical fold split (`random_state = 1000*seed + repeat`). Per
repeat: a single 5-fold out-of-fold pass in which each galaxy is predicted exactly once; all
metrics, including prec@50, come from that one pass. The harness returns one row per
(plane, strength, model, scenario, seed), so every paired comparison downstream is exact by
construction.

In [ ]:
def _metrics_on(build, d, feats, tgt, seed, n_repeats=N_REPEATS, tag=""):
    """Averaged metrics for ONE model on ONE already-built catalog."""
    X  = d[feats].to_numpy()
    y  = d[tgt].to_numpy()
    pt = d["p_true"].to_numpy()

    reps = []
    for r in range(n_repeats):
        skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=1000 * seed + r)
        oof = np.full(len(y), np.nan)
        for tr, te in skf.split(X, y):
            # assert rather than silently skip: a dropped fold changes the
            # denominator and biases the mean toward the lucky folds.
            # Low strength thins the positives, so this can fire where it never did at 3.0.
            assert y[te].sum() >= 1, f"degenerate fold: {tag} seed={seed} repeat={r}"
            oof[te] = build().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        assert not np.isnan(oof).any(), f"some galaxy was never predicted: {tag}"

        top = np.argsort(-oof)[:N_AT]
        reps.append({
            "auc":                 roc_auc_score(y, oof),
            "pr_auc":              average_precision_score(y, oof),
            "brier":               float(np.mean((oof - y) ** 2)),
            "mse_vs_truth":        float(np.mean((oof - pt) ** 2)),
            "prec_at_n":           float(y[top].mean()),
            "promised_at_n":       float(oof[top].sum()),
            "delivered_at_n":      float(y[top].sum()),
            "promised_total":      float(oof.sum()),
            "actual_total":        float(y.sum()),
            "true_expected_total": float(pt.sum()),
        })

    out = pd.DataFrame(reps).mean().to_dict()
    out["seed"] = seed
    out["n"]    = len(y)         # catalog size, for prevalence baselines later
    return out

def _eval_models_one_seed(builders, ds_fn, feats, tgt, seed, tag=""):
    """All models on the SAME catalog. One dataset build per (plane, strength,
    scenario, seed) instead of one per model. Top level so joblib can ship it."""
    d = ds_fn(seed)
    return {name: _metrics_on(b, d, feats, tgt, seed, tag=f"{tag}/{name}")
            for name, b in builders.items()}

def run_plane_strength(plane, strength, seeds=None):
    """Tidy per-seed results for one (plane, strength). No aggregation here."""
    seeds = list(seeds) if seeds is not None else report_seeds()
    models = TUNED[strength][plane]
    feats, tgt = PLANE_COLS[plane]
    tasks = [(sc, s) for sc in SCENARIOS for s in seeds]
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(_eval_models_one_seed)(
            models, dataset_fn(plane, sc, strength), feats, tgt, s,
            f"{plane}/s={strength}/{sc}")
        for sc, s in tasks
    )
    rows = [{"plane": plane, "strength": strength, "model": name,
             "scenario": sc, **m}
            for (sc, _), per_model in zip(tasks, res)
            for name, m in per_model.items()]
    print(f"  {plane:5s} strength={strength}: {len(models)} models x {len(SCENARIOS)} "
          f"scenarios x {len(seeds)} seeds on {len(tasks)} shared catalogs "
          f"in {time.time() - t0:.0f}s", flush=True)
    return pd.DataFrame(rows)

def run_all(planes=PLANES, strengths=STRENGTHS, seeds=None):
    t0 = time.time()
    out = pd.concat([run_plane_strength(p, st, seeds)
                     for p in planes for st in strengths],
                    ignore_index=True)
    print(f"total {time.time() - t0:.0f}s")
    return out

# ---- aggregation and display: every function takes a per-seed frame ----
def sub(R, plane, strength):
    return R[(R["plane"] == plane) & (R["strength"] == strength)]

def _models_in(df):
    present = list(df["model"].unique())
    return [m for m in MODEL_ORDER if m in present] + \
           [m for m in present if m not in MODEL_ORDER]

def _scenario_mean_per_seed(per_seed, metric):
    """[seed x model]: scenario-mean of the metric, one row per seed. Pairing preserved."""
    return per_seed.groupby(["seed", "model"])[metric].mean().unstack("model")

def summarize(per_seed, metric):
    """Mean and standard error over seeds, per (model, scenario). The MEAN column is the
    scenario-mean taken within each seed first, so its SE is exact rather than a pooled
    approximation that assumes the scenarios are independent."""
    g    = per_seed.groupby(["model", "scenario"])[metric]
    mean = g.mean().unstack("scenario")[SCENARIOS]
    se   = (g.std(ddof=1) / np.sqrt(g.count())).unstack("scenario")[SCENARIOS]
    sm   = _scenario_mean_per_seed(per_seed, metric)
    mean["MEAN"] = sm.mean().reindex(mean.index)
    se["MEAN"]   = (sm.std(ddof=1) / np.sqrt(sm.count())).reindex(mean.index)
    order = mean["MEAN"].sort_values(ascending=not HIGHER_BETTER[metric]).index
    return mean.loc[order].round(3), se.loc[order].round(3)

def show_metric(per_seed, metric, title=""):
    role = "PRIMARY" if metric in PRIMARY_METRICS else "diagnostic"
    m, e = summarize(per_seed, metric)
    print(f"===== {title}{metric} ({role}; higher better: {HIGHER_BETTER[metric]}) "
          f"- mean +/- SE over {per_seed['seed'].nunique()} seeds =====")
    display(m.astype(str) + " +/- " + e.astype(str))

def _diff_series(per_seed, metric, hi, lo):
    """Per-(scenario, seed) difference hi - lo. The catalogs cancel, so the SE is far smaller
    than on either raw mean."""
    a = per_seed[per_seed["model"] == hi].set_index(["scenario", "seed"])[metric]
    b = per_seed[per_seed["model"] == lo].set_index(["scenario", "seed"])[metric]
    d = (a - b).dropna()
    assert len(d) == len(a) == len(b), f"unpaired rows in {hi} - {lo}"
    return d

def paired_table(per_seed, metric, baseline=BASELINE_MODEL, models=None):
    """Per-scenario paired difference against a baseline model."""
    rows = []
    for name in _models_in(per_seed):
        if name == baseline or (models is not None and name not in models):
            continue
        gm = _diff_series(per_seed, metric, name, baseline).groupby("scenario")
        mean, se, n = gm.mean(), gm.std(ddof=1) / np.sqrt(gm.count()), gm.count()
        for sc in SCENARIOS:
            rows.append({"model": name, "scenario": sc,
                         "diff": mean[sc], "se": se[sc],
                         "t": mean[sc] / se[sc] if se[sc] > 0 else np.nan,
                         "n": int(n[sc])})
    return pd.DataFrame(rows)

def pooled_diff(per_seed, metric, hi, lo):
    """Scenario-mean difference taken within a seed, then SE over seeds. Exact under seed
    independence, unlike summing per-scenario SEs as if the scenarios were independent."""
    d = _diff_series(per_seed, metric, hi, lo).groupby("seed").mean()
    mean, se = d.mean(), d.std(ddof=1) / np.sqrt(len(d))
    return mean, se, (mean / se if se > 0 else np.nan), len(d)

def _cells(t):
    return (t["diff"].round(3).astype(str) + " +/- " + t["se"].round(3).astype(str)
            + t["t"].apply(lambda v: "  *" if np.isfinite(v) and abs(v) >= 2 else ""))

def show_paired(per_seed, metric, baseline=BASELINE_MODEL, title=""):
    t = paired_table(per_seed, metric, baseline)
    out = (t.assign(cell=_cells(t))
            .pivot(index="model", columns="scenario", values="cell")[SCENARIOS])
    out = out.reindex([m for m in MODEL_ORDER if m in out.index])
    print(f"===== {title}{metric}: paired difference vs {baseline} "
          f"(higher better: {HIGHER_BETTER[metric]}; * = |t| >= 2) =====")
    display(out)

def ladder_table(per_seed, metric):
    """The LR complexity ladder: each rung against the rung below it. A positive, resolvable
    step is the evidence that the extra terms pay off."""
    rows = []
    for hi, lo in LADDER:
        if hi not in per_seed["model"].values or lo not in per_seed["model"].values:
            continue
        mean, se, t, n = pooled_diff(per_seed, metric, hi, lo)
        per_sc = paired_table(per_seed, metric, baseline=lo, models=[hi])
        rows.append({"step": f"{hi} - {lo}", "diff": mean, "se": se, "t": t,
                     "seeds": n,
                     "scenarios_with_|t|>=2": int((per_sc["t"].abs() >= 2).sum())})
    return pd.DataFrame(rows).set_index("step").round(4)

def promise_topn_table(per_seed):
    a = per_seed.groupby(["model", "scenario"])[["promised_at_n", "delivered_at_n"]].mean()
    p = a["promised_at_n"].unstack("scenario")[SCENARIOS]
    d = a["delivered_at_n"].unstack("scenario")[SCENARIOS]
    out = pd.concat({"promised": p, "delivered": d}, axis=1).swaplevel(axis=1)
    return out.reindex(columns=pd.MultiIndex.from_product(
        [SCENARIOS, ["promised", "delivered"]])).round(1)

def promise_total_table(per_seed):
    t = (per_seed.groupby(["model", "scenario"])["promised_total"].mean()
         .unstack("scenario")[SCENARIOS])
    t.loc["ACTUAL masers"]       = per_seed.groupby("scenario")["actual_total"].mean()[SCENARIOS]
    t.loc["ORACLE (sum p_true)"] = per_seed.groupby("scenario")["true_expected_total"].mean()[SCENARIOS]
    return t.round(1)

## 5. Data overview

Context figures only. Nothing here feeds the evaluation. The scenario panel is drawn once per
strength, at the strength the models are actually scored on.

In [ ]:
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "font.family": "DejaVu Sans",
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.linewidth": .9, "axes.edgecolor": "#3A3A3A",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "savefig.bbox": "tight",
    "savefig.facecolor": "white", "figure.facecolor": "white",
})
from matplotlib.lines import Line2D

NEG, POS, REAL_POS = "#B9C2CC", "#C2452D", "#1F5C8B"
PLANE_LABELS = {
    "xray": ("X-ray plane", "log L_12um", "log L_obs (2-10 keV)"),
    "wise": ("WISE plane",  "W1 - W2",    "W2 - W3"),
}
PLANE_TITLES = {"xray": "X-ray Plane", "wise": "WISE Plane"}

SAVE_FIGS = True
FIG_DIR   = "poster_figs"
if SAVE_FIGS:
    os.makedirs(FIG_DIR, exist_ok=True)

def _save(fig, name):
    if SAVE_FIGS:
        path = os.path.join(FIG_DIR, f"{name}.png")
        fig.savefig(path, dpi=300, bbox_inches="tight")
        print("saved", path)

SEED_FIG = 5
for ST in STRENGTHS:
    fig, axes = plt.subplots(len(PLANES), len(SCENARIOS), figsize=(16, 7.0),
                             constrained_layout=True, squeeze=False)
    for r, plane in enumerate(PLANES):
        row_title, xlab, ylab = PLANE_LABELS[plane]
        fx, fy = FEATURES[plane]
        tgt = TARGET[plane]
        xlim = ylim = None
        for c, sc in enumerate(SCENARIOS):
            ax = axes[r, c]
            df = make_catalog(plane, sc, ST, SEED_FIG)
            neg, pos = df[df[tgt] == 0], df[df[tgt] == 1]
            ax.scatter(neg[fx], neg[fy], s=7, c=NEG, alpha=.55, lw=0, rasterized=True)
            ax.scatter(pos[fx], pos[fy], s=22, c=POS, alpha=.9, lw=.4,
                       edgecolors="white", rasterized=True)
            if r == 0:
                ax.set_title(sc.capitalize(), pad=8, fontweight="bold")
            if c == 0:
                ax.set_ylabel(f"{row_title}\n{ylab}", labelpad=8)
            else:
                ax.tick_params(labelleft=False)
            ax.set_xlabel(xlab)
            ax.grid(True, lw=.4, color="#E6E6E6"); ax.set_axisbelow(True)
            ax.annotate(f"{len(pos)} pos / {len(df)}", xy=(.03, .93),
                        xycoords="axes fraction", fontsize=8.5, color="#5A5A5A")
            if c == 0:
                xlim, ylim = ax.get_xlim(), ax.get_ylim()
            else:
                ax.set_xlim(xlim); ax.set_ylim(ylim)
    handles = [Line2D([], [], marker="o", ls="", ms=7, mfc=POS, mec="white", label="Simulated maser"),
               Line2D([], [], marker="o", ls="", ms=6, mfc=NEG, mec="none", label="Non-detection")]
    fig.legend(handles=handles, loc="lower center", ncol=2,
               bbox_to_anchor=(.5, -.035), fontsize=11)
    fig.suptitle(f"Synthetic catalogs under five true boundary scenarios "
                 f"(strength={ST}, ceiling=0.4)", fontsize=15, fontweight="bold")
    _save(fig, f"scenario_panel_s{ST}"); plt.show()

In [ ]:
# ---- real catalogs: CONTEXT ONLY ----
# No model in this notebook is fit on real data. This figure exists to show
# why the synthetic prevalence and overlap were chosen. Set to False to skip.
SHOW_REAL_CONTEXT = True

if SHOW_REAL_CONTEXT:
    from maser_data import load_xray_sample, load_wise
    real = {
        "xray": (load_xray_sample(), "X-ray sample", *PLANE_LABELS["xray"][1:]),
        "wise": (load_wise(),        "WISE sample",  *PLANE_LABELS["wise"][1:]),
    }
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), constrained_layout=True)
    for ax, (plane, (df, title, xlab, ylab)) in zip(axes, real.items()):
        fx, fy = FEATURES[plane]
        tgt = TARGET[plane]
        d = df.dropna(subset=[fx, fy, tgt])
        neg, pos = d[d[tgt] == 0], d[d[tgt] == 1]
        ax.scatter(neg[fx], neg[fy], s=8, c=NEG, alpha=.55, lw=0, rasterized=True)
        ax.scatter(pos[fx], pos[fy], s=26, c=REAL_POS, alpha=.9, lw=.4,
                   edgecolors="white", rasterized=True)
        ax.set_title(f"{title}  ({len(pos)} masers / {len(d)} galaxies, "
                     f"{100*len(pos)/len(d):.1f}%)", pad=8, fontweight="bold")
        ax.set_xlabel(xlab); ax.set_ylabel(ylab)
        ax.grid(True, lw=.4, color="#E6E6E6"); ax.set_axisbelow(True)
    axes[1].set_xlim(-0.6, 1.8); axes[1].set_ylim(0, 6)
    handles = [Line2D([], [], marker="o", ls="", ms=7, mfc=REAL_POS, mec="white", label="Known maser"),
               Line2D([], [], marker="o", ls="", ms=6, mfc=NEG, mec="none", label="Searched, not detected")]
    fig.legend(handles=handles, loc="lower center", ncol=2,
               bbox_to_anchor=(.5, -.06), fontsize=11)
    fig.suptitle("Observed data (context only - no model is fit on this)",
                 fontsize=15, fontweight="bold")
    _save(fig, "real_context"); plt.show()

## 6. Run the grid

Two planes x three strengths x five scenarios x forty seeds, five models on every shared catalog.
WISE (n=4400) dominates. Use `SEED_LIMIT` for a rehearsal run.

Results are written to `per_seed_results.csv` so sections 7-9 can be re-run, re-cut or re-plotted
without paying for the grid again.

In [ ]:
R = run_all()
R.to_csv("per_seed_results.csv", index=False)
print(R.shape, "rows ->", os.path.abspath("per_seed_results.csv"))
display(R.head())

# To skip the grid on a later session:
# R = pd.read_csv("per_seed_results.csv")

### 6.1 Realized prevalence

The n=50 cut is fixed, so a strength that changes the positive rate also moves what a random
ranking scores. Compare `prec_at_n` levels across strengths only against these numbers, or use
the skill-style views (paired differences vs the baseline, Brier skill score) which are already
prevalence-referenced.

In [ ]:
PREV = {(p, st): (g["actual_total"] / g["n"]).groupby(g["scenario"]).mean().reindex(SCENARIOS)
        for (p, st), g in R.groupby(["plane", "strength"])}

prev_tbl = pd.DataFrame(PREV).T
prev_tbl.index.names = ["plane", "strength"]
print("realized positive rate by plane / strength / scenario (real surveys run 2.7-8.3%):")
display(prev_tbl.round(4))

print("\nrandom-ranking yield in the top-50 (= 50 * prevalence):")
display((prev_tbl * N_AT).round(2))

## 7. Per-(plane, strength) results

Primary metrics, then the paired comparison against `lr_linear`, then the LR complexity ladder,
then the diagnostics and the promised-vs-delivered tables.

In [ ]:
for plane in PLANES:
    for st in STRENGTHS:
        s = sub(R, plane, st)
        tag = f"[{plane} s={st}] "
        print("#" * 78)
        print(f"# {plane.upper()} plane, strength={st}   ({SPEC_SOURCE[st]})")
        print("#" * 78)
        for m in PRIMARY_METRICS:
            show_metric(s, m, title=tag)
        for m in PRIMARY_METRICS:
            show_paired(s, m, title=tag)
        for m in PRIMARY_METRICS:
            print(f"===== {tag}{m}: LR complexity ladder, pooled over scenarios "
                  f"(higher better: {HIGHER_BETTER[m]}) =====")
            display(ladder_table(s, m))
        for m in DIAGNOSTIC_METRICS:
            show_metric(s, m, title=tag)
        print(f"{tag}top-{N_AT}: masers promised vs delivered")
        display(promise_topn_table(s))
        print(f"\n{tag}whole sample: promised vs actual vs oracle")
        display(promise_total_table(s))
        print("\n")

## 8. Does the ranking hold across strengths?

Four views, weakest to strongest:

1. **Rank table** - order the models by the scenario-mean of each primary metric, per
   (plane, strength). Ranks hide how close the models are, so this is a summary, not evidence.
2. **Kendall tau** between strengths, per plane. Tau of 1.0 means identical ordering.
3. **Paired difference vs `lr_linear`** side by side across strengths. This is the view that
   carries error bars, and it is the one to quote.
4. **Seed bootstrap** - resample the 40 seeds jointly across models (preserving pairing) and
   record how often each model comes out on top. It answers "how load-bearing is the winner",
   which the point rank cannot. The scenario set is held fixed, so this covers seed noise only,
   not scenario-choice uncertainty.

Winner flips between strengths only matter if the gaps involved are resolvable. Read 3 and 4
before claiming either stability or instability.

In [ ]:
def scenario_mean_by_seed(R, plane, strength, metric):
    """[seed x model] table of the scenario-mean metric. Pairing preserved."""
    piv = _scenario_mean_per_seed(sub(R, plane, strength), metric)
    return piv[[m for m in MODEL_ORDER if m in piv.columns]]

def rank_table(R, metric):
    rows = {}
    for plane in PLANES:
        for st in STRENGTHS:
            mean = scenario_mean_by_seed(R, plane, st, metric).mean()
            rows[(plane, st)] = mean.rank(ascending=not HIGHER_BETTER[metric])
    out = pd.DataFrame(rows).astype(int)
    out.columns.names = ["plane", "strength"]
    return out

def mean_table(R, metric):
    rows = {(plane, st): scenario_mean_by_seed(R, plane, st, metric).mean()
            for plane in PLANES for st in STRENGTHS}
    out = pd.DataFrame(rows)
    out.columns.names = ["plane", "strength"]
    return out.round(4)

for m in PRIMARY_METRICS:
    print(f"===== {m}: scenario-mean value (higher better: {HIGHER_BETTER[m]}) =====")
    display(mean_table(R, m))
    print(f"===== {m}: rank (1 = best) =====")
    display(rank_table(R, m))

In [ ]:
from scipy.stats import kendalltau

def rank_agreement(R, metric):
    rt = rank_table(R, metric)
    rows = []
    for plane in PLANES:
        for a, b in itertools.combinations(STRENGTHS, 2):
            tau, p = kendalltau(rt[(plane, a)], rt[(plane, b)])
            rows.append({"plane": plane, "pair": f"{a} vs {b}",
                         "kendall_tau": round(tau, 3), "p": round(p, 4),
                         "same_winner": rt[(plane, a)].idxmin() == rt[(plane, b)].idxmin(),
                         "winner_a": rt[(plane, a)].idxmin(),
                         "winner_b": rt[(plane, b)].idxmin()})
    return pd.DataFrame(rows)

for m in PRIMARY_METRICS:
    print(f"===== {m}: rank agreement between strengths "
          f"(tau over {len(MODEL_ORDER)} models - low power, treat as descriptive) =====")
    display(rank_agreement(R, m))

In [ ]:
def paired_across_strengths(R, plane, metric, baseline=BASELINE_MODEL):
    """Paired diff vs baseline, pooled over scenarios within a seed, one column per strength."""
    cols = {}
    for st in STRENGTHS:
        s = sub(R, plane, st)
        rows = {}
        for mdl in _models_in(s):
            if mdl == baseline:
                continue
            mean, se, t, _ = pooled_diff(s, metric, mdl, baseline)
            rows[mdl] = (f"{mean:.3f} +/- {se:.3f}"
                         + ("  *" if np.isfinite(t) and abs(t) >= 2 else ""))
        cols[st] = pd.Series(rows)
    out = pd.DataFrame(cols)
    out.columns.name = "strength"
    return out.reindex([m for m in MODEL_ORDER if m in out.index])

for plane in PLANES:
    for m in PRIMARY_METRICS:
        print(f"===== {plane}: {m}, paired difference vs {BASELINE_MODEL}, pooled over "
              f"scenarios (higher better: {HIGHER_BETTER[m]}; * = |t| >= 2) =====")
        display(paired_across_strengths(R, plane, m))

In [ ]:
def bootstrap_best(R, plane, strength, metric, n_boot=4000, seed=0):
    """P(model is best) under resampling of seeds. Seeds are resampled jointly
    across models, so the pairing that makes small gaps resolvable is preserved."""
    piv = scenario_mean_by_seed(R, plane, strength, metric)
    A, models = piv.to_numpy(), list(piv.columns)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, A.shape[0], size=(n_boot, A.shape[0]))
    means = A[idx].mean(axis=1)
    best = means.argmax(1) if HIGHER_BETTER[metric] else means.argmin(1)
    return pd.Series(np.bincount(best, minlength=len(models)) / n_boot, index=models)

for m in PRIMARY_METRICS:
    tbl = pd.DataFrame({(plane, st): bootstrap_best(R, plane, st, m)
                        for plane in PLANES for st in STRENGTHS})
    tbl.columns.names = ["plane", "strength"]
    print(f"===== {m}: P(best model), seed bootstrap, 4000 resamples =====")
    display(tbl.round(3))

In [ ]:
# LR ladder across strengths: does the quadratic step keep paying as the data get harder?
for plane in PLANES:
    for m in PRIMARY_METRICS:
        cols = {}
        for st in STRENGTHS:
            lt = ladder_table(sub(R, plane, st), m)
            cols[st] = (lt["diff"].round(4).astype(str) + " +/- "
                        + lt["se"].round(4).astype(str)
                        + lt["t"].apply(lambda v: "  *" if np.isfinite(v) and abs(v) >= 2 else ""))
        out = pd.DataFrame(cols); out.columns.name = "strength"
        print(f"===== {plane}: {m}, LR complexity ladder by strength "
              f"(higher better: {HIGHER_BETTER[m]}; * = |t| >= 2) =====")
        display(out)

## 9. Figures

Error bars are **+/- 1 SE over report seeds**. Dotted line is the catalog's own positive rate at
that (plane, strength), which is what a random ranking scores in expectation.

Real-survey context for captions, not plotted: Kuo et al. 2018 GBT sample, 2.7% megamasers and
0.9% disk masers; Kuo et al. 2020 X-ray sample, 8.3% megamasers (53/641) and 3.7% disk (24/641).

In [ ]:
MODEL_COLORS = {"lr_linear": "#4C72B0", "lr_interaction": "#DD8452",
                "lr_quadratic": "#55A868", "rf": "#C44E52", "hgb": "#8172B3"}
PROMISE_COLOR, DELIVER_COLOR = "#B0B0B0", "#2A6F97"
RAND_COLOR, RAND_STYLE, HALF = "#333333", (0, (1.6, 1.6)), 0.44

plt.rcParams.update({"font.size": 17, "axes.titlesize": 18, "axes.labelsize": 17,
                     "xtick.labelsize": 15, "ytick.labelsize": 15,
                     "legend.fontsize": 15})

def _mean_se(res, metric):
    g = res.groupby(["model", "scenario"])[metric]
    return (g.mean().unstack("scenario")[SCENARIOS],
            (g.std(ddof=1) / np.sqrt(g.count())).unstack("scenario")[SCENARIOS])

def _bar_x(i, j, n):
    w = 0.8 / n
    return i + j * w - 0.4 + w / 2, w

def _label_bars(ax, xs, h, e, fmt="{:.2f}", fontsize=7.5, rot=90):
    pad = 0.015 * (ax.get_ylim()[1] - ax.get_ylim()[0])
    for x, hh, ee in zip(xs, h, e):
        ee = 0.0 if not np.isfinite(ee) else ee
        ax.text(x, hh + ee + pad, fmt.format(hh), ha="center", va="bottom",
                fontsize=fontsize, rotation=rot)

def _draw_baselines(ax, level, x):
    for i, sc in enumerate(SCENARIOS):
        ax.hlines(level[sc], x[i] - HALF, x[i] + HALF, colors=RAND_COLOR,
                  linestyles=RAND_STYLE, linewidth=1.8, zorder=6)

def _baseline_handle(label):
    return [plt.Line2D([0], [0], color=RAND_COLOR, linestyle=RAND_STYLE,
                       linewidth=1.8, label=label)]

def _legend_below(ax, extra, ncol=3):
    ax.legend(handles=ax.get_legend_handles_labels()[0] + extra,
              loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=ncol,
              fontsize=10, frameon=False, handlelength=1.8, columnspacing=1.4)

def _grouped_bars(res, values, errs, models, ylabel, title, tag,
                  baseline_level=None, baseline_label=None, hline0=False, fmt="{:.2f}"):
    # both callers pass higher-is-better quantities (prec@N, BSS), so best = max
    x = np.arange(len(SCENARIOS))
    fig, ax = plt.subplots(figsize=(9, 6.2))
    bars = {}
    for j, mdl in enumerate(models):
        xs, w = _bar_x(x, j, len(models))
        bars[mdl] = ax.bar(xs, values[mdl].values, w, yerr=errs[mdl].values, capsize=2,
                           color=MODEL_COLORS.get(mdl, "#777"),
                           edgecolor="black", linewidth=0.4)
    lo = float(np.nanmin(np.minimum(values.values - errs.values, 0)))
    hi = float(np.nanmax(values.values + errs.values))
    ax.set_ylim(min(0, lo * 1.15), hi * 1.30)
    for j, mdl in enumerate(models):
        xs, _ = _bar_x(x, j, len(models))
        _label_bars(ax, xs, values[mdl].values, errs[mdl].values, fmt=fmt)

    # gold outline on the best model in each scenario
    for i, scen in enumerate(SCENARIOS):
        col = values.iloc[i]
        if col.isna().all():
            continue
        patch = bars[col.idxmax()][i]
        patch.set_edgecolor("gold"); patch.set_linewidth(2.6); patch.set_zorder(4)

    model_handles = [plt.Rectangle((0, 0), 1, 1, facecolor=MODEL_COLORS.get(m, "#777"),
                                   edgecolor="black", linewidth=0.4, label=m)
                     for m in models]
    best_box_proxy = plt.Rectangle((0, 0), 1, 1, facecolor="#DDDDDD",
                                   edgecolor="gold", linewidth=2.2, label="best in scenario")
    if baseline_level is not None:
        _draw_baselines(ax, baseline_level, x)
    if hline0:
        ax.axhline(0, color=RAND_COLOR, linestyle=RAND_STYLE, linewidth=1.8, zorder=6)
    ax.set_xticks(x); ax.set_xticklabels(SCENARIOS)
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.text(0.98, 0.98, f"Best Overall: {values.mean(axis=0).idxmax()}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow",
                      edgecolor="gold", linewidth=1.2))
    _legend_below(ax, model_handles + _baseline_handle(baseline_label) + [best_box_proxy])
    fig.tight_layout(); fig.subplots_adjust(bottom=0.26)
    _save(fig, tag); plt.show()

def plot_prec_by_scenario(res, plane, strength, prev):
    models = _models_in(res)
    t, e = _mean_se(res, "prec_at_n")
    _grouped_bars(res, t.loc[models].T, e.loc[models].T, models,
                  f"precision@{N_AT}",
                  f"{PLANE_TITLES[plane]}, strength={strength}\n"
                  f"(+/- 1 SE over {res['seed'].nunique()} seeds)",
                  f"{plane}_s{strength}_prec_by_scenario",
                  baseline_level=prev, baseline_label="random ranking (catalog rate)")

def plot_brier_skill_by_scenario(res, plane, strength, prev):
    models = _models_in(res)
    t, e = _mean_se(res, "brier")
    t, e = t.loc[models].T, e.loc[models].T
    null = pd.Series({sc: prev[sc] * (1 - prev[sc]) for sc in SCENARIOS}).reindex(SCENARIOS)
    _grouped_bars(res, 1 - t.div(null, axis=0), e.div(null, axis=0), models,
                  "Brier skill score",
                  f"{PLANE_TITLES[plane]}, strength={strength}\n"
                  f"(+/- 1 SE over {res['seed'].nunique()} seeds)",
                  f"{plane}_s{strength}_bss_by_scenario",
                  hline0=True, baseline_label="no-skill baseline (BSS = 0)")

def plot_pvd_by_scenario(res, plane, strength, prev):
    models = _models_in(res)
    pm, pe = _mean_se(res, "promised_at_n")
    dm, de = _mean_se(res, "delivered_at_n")
    pm, pe, dm, de = (df.loc[models].T for df in (pm, pe, dm, de))
    x = np.arange(len(models)); w = 0.38
    rand_yield = {sc: N_AT * prev[sc] for sc in SCENARIOS}
    ceiling = float(np.nanmax([np.nanmax(pm.values + pe.values),
                               np.nanmax(dm.values + de.values),
                               max(rand_yield.values())]))
    fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(16, 5.2), sharey=True)
    for ax, sc in zip(axes, SCENARIOS):
        ax.bar(x - w/2, pm.loc[sc].values, w, yerr=pe.loc[sc].values, capsize=2,
               color=PROMISE_COLOR, edgecolor="black", linewidth=0.4)
        ax.bar(x + w/2, dm.loc[sc].values, w, yerr=de.loc[sc].values, capsize=2,
               color=DELIVER_COLOR, edgecolor="black", linewidth=0.4)
        ax.set_ylim(0, ceiling * 1.30)
        _label_bars(ax, x - w/2, pm.loc[sc].values, pe.loc[sc].values, fmt="{:.1f}", fontsize=7)
        _label_bars(ax, x + w/2, dm.loc[sc].values, de.loc[sc].values, fmt="{:.1f}", fontsize=7)
        ax.axhline(rand_yield[sc], color=RAND_COLOR, linestyle=RAND_STYLE,
                   linewidth=1.8, zorder=6)
        ax.set_title(sc); ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=90, fontsize=9)
    axes[0].set_ylabel(f"masers in top-{N_AT}")
    handles = [plt.Rectangle((0, 0), 1, 1, color=PROMISE_COLOR),
               plt.Rectangle((0, 0), 1, 1, color=DELIVER_COLOR)]
    bl = _baseline_handle(f"random {N_AT} targets (catalog rate)")
    fig.legend(handles + bl, ["promised", "delivered", bl[0].get_label()],
               loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.10), fontsize=10)
    fig.suptitle(f"{PLANE_TITLES[plane]}, strength={strength}: promised vs delivered "
                 f"in top-{N_AT} (error bars = +/- 1 SE over seeds)", y=1.02)
    plt.tight_layout(); _save(fig, f"{plane}_s{strength}_pvd_by_scenario"); plt.show()

def plot_pvd_calibration(res, plane, strength):
    models = _models_in(res)
    markers = ["o", "s", "^", "D", "v"]
    a = res.groupby(["model", "scenario"])[["promised_at_n", "delivered_at_n"]].mean()
    fig, ax = plt.subplots(figsize=(6, 6))
    allv = []
    for mdl in models:
        for k, sc in enumerate(SCENARIOS):
            p, d = a.loc[(mdl, sc), "promised_at_n"], a.loc[(mdl, sc), "delivered_at_n"]
            ax.scatter(d, p, color=MODEL_COLORS.get(mdl, "#777"),
                       marker=markers[k % len(markers)], s=80,
                       edgecolor="black", linewidth=0.4, zorder=3)
            allv += [p, d]
    lo, hi = min(allv) * 0.9, max(allv) * 1.05
    ax.plot([lo, hi], [lo, hi], "--", color="gray", label="promise met (y = x)")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(f"delivered masers (top-{N_AT})")
    ax.set_ylabel(f"promised masers (top-{N_AT})")
    ax.set_title(f"{PLANE_TITLES[plane]}, strength={strength}: Promised vs. Delivered")
    mh = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=MODEL_COLORS.get(m, "#777"),
                     markersize=10, label=m) for m in models]
    sh = [plt.Line2D([0], [0], marker=markers[i], color="gray", linestyle="",
                     markersize=8, label=sc) for i, sc in enumerate(SCENARIOS)]
    ax.legend(handles=mh + sh, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    plt.tight_layout(); _save(fig, f"{plane}_s{strength}_pvd_calibration"); plt.show()

### 9.1 The strength-sweep figure

One panel per plane per primary metric: model performance against strength, scenario-averaged,
with +/- 1 SE bands over seeds. This is the figure that answers "does the ranking hold" at a
glance; the tables in section 8 are what back it up.

In [ ]:
def plot_metric_vs_strength(R, metric):
    fig, axes = plt.subplots(1, len(PLANES), figsize=(6.2 * len(PLANES), 5.0), squeeze=False)
    for ax, plane in zip(axes[0], PLANES):
        for mdl in MODEL_ORDER:
            mu, se = [], []
            for st in STRENGTHS:
                col = scenario_mean_by_seed(R, plane, st, metric)
                if mdl not in col.columns:
                    mu.append(np.nan); se.append(np.nan); continue
                v = col[mdl]
                mu.append(v.mean()); se.append(v.std(ddof=1) / np.sqrt(v.count()))
            mu, se = np.array(mu), np.array(se)
            ax.plot(STRENGTHS, mu, marker="o", label=mdl, color=MODEL_COLORS.get(mdl, "#777"))
            ax.fill_between(STRENGTHS, mu - se, mu + se, alpha=0.15,
                            color=MODEL_COLORS.get(mdl, "#777"))
        ax.set_xticks(STRENGTHS)
        ax.set_xlabel("strength"); ax.set_ylabel(metric)
        ax.set_title(PLANE_TITLES[plane])
    axes[0][0].legend(fontsize=10)
    fig.suptitle(f"{metric} vs separability (scenario-averaged, shaded band = +/- 1 SE over seeds)",
                 y=1.02, fontsize=14)
    plt.tight_layout(); _save(fig, f"{metric}_vs_strength"); plt.show()

for m in PRIMARY_METRICS:
    plot_metric_vs_strength(R, m)

### 9.2 Per-scenario figures

Six prec@50 panels and six BSS panels (two planes x three strengths). Set
`MAKE_PVD_FIGS = False` to skip the promised-vs-delivered pair.

In [ ]:
MAKE_PVD_FIGS = True

for plane in PLANES:
    for st in STRENGTHS:
        res = sub(R, plane, st)
        prev = PREV[(plane, st)]
        plot_prec_by_scenario(res, plane, st, prev)
        plot_brier_skill_by_scenario(res, plane, st, prev)
        if MAKE_PVD_FIGS:
            plot_pvd_by_scenario(res, plane, st, prev)
            plot_pvd_calibration(res, plane, st)

## 10. Summary

**What this notebook runs.** Five models - a plain logistic regression (`lr_linear`), an
interaction-only logistic regression (`lr_interaction`), a quadratic logistic regression
(`lr_quadratic`), a random forest, and sklearn's histogram gradient boosting - on two feature
planes (X-ray, WISE) at three separability settings (strength 1.0, 2.0, 3.0). Hyperparameters
come from the selection notebook, which sweeps each of the five models independently at each
strength on seeds 0-4; every number here is computed on seeds 5-44. The fused plane and the
`wise_signal` sweep are not part of this notebook.

**Evaluation loop.** For each (plane, strength, scenario, seed), one catalog is built and reused
by all five models, so every comparison is paired by construction (same galaxies, same folds).
5-fold stratified CV, fold seed `1000*seed + repeat`, 1 repeat. Predictions are collected
out-of-fold into a single array, with asserts against degenerate folds and unpredicted rows; the
degenerate-fold assert is more likely to fire at strength 1.0 than at 3.0, and it names the
plane, strength and scenario when it does. Metrics are computed once on the full OOF array, never
per-fold. The selection notebook uses this same harness, so a config's selection numbers and its
held-out numbers are the same quantity measured on different seeds.

**Metrics.** Primary: `prec_at_n` (precision in the top 50) and `mse_vs_truth` (squared error
against the generator's true probability, valid because `p_obs == p_true` is asserted at every
strength). Diagnostic: Brier, PR-AUC, ROC-AUC. Also tracked: promised vs actual vs oracle counts.

**Cross-strength reading.** Section 8 gives four views: rank table, Kendall tau between strengths,
paired differences vs `lr_linear` with standard errors side by side across strengths, and a seed
bootstrap giving P(best) per model. The paired table and the bootstrap are the load-bearing ones.
Ranks flip easily when models are separated by less than their standard errors, so a flipped rank
is not by itself evidence of instability, and an unflipped rank is not by itself evidence of
stability.

**Caveats to keep attached to any claim made from this run.**

1. Selection ranks on `prec_at_n` alone. Where the selection notebook's metric-sensitivity table
   shows `mse_vs_truth` would have picked a different config, the calibration numbers here are
   for a config chosen on ranking.
2. `prec_at_n` is a fixed-n cut, so its level moves with the realized positive rate, which
   strength can change. Section 6.1 prints that rate. Compare paired differences and Brier skill
   scores across strengths; compare raw `prec_at_n` levels only within a (plane, strength).
3. The bootstrap resamples seeds only. The five scenarios are held fixed, so none of this covers
   uncertainty about the choice of scenario set.